# Trustworthy Personalised AI — Analysis Dashboard

Research notebook for analysing training data quality, model benchmark results, and conversation samples. All charts are saved to `exports/` as SVG (vector, dissertation-ready) and PNG (high-resolution raster) via plotly + kaleido. Run cells top-to-bottom on first use; individual sections can be re-run independently after that.

In [ ]:
import json
import re
from pathlib import Path
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
from IPython.display import HTML, display

pio.templates.default = "plotly_white"
PALETTE = px.colors.qualitative.Set2

DATA_DIR    = Path("data")
REPORTS_DIR = Path("reports")
EXPORTS_DIR = Path("exports")
EXPORTS_DIR.mkdir(exist_ok=True)

def save_fig(fig, name):
    """Save dissertation-ready SVG + high-res PNG and show inline."""
    fig.write_image(str(EXPORTS_DIR / f"{name}.svg"))
    fig.write_image(str(EXPORTS_DIR / f"{name}.png"), scale=3)
    print(f"\u2713 exports/{name}.svg + .png")
    fig.show()

## Section 1 — Data Loading

In [ ]:
def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

def extract_tag_len(content, tag):
    """Return total character length of ALL <tag>\u2026</tag> blocks, else 0."""
    return sum(len(m.strip()) for m in re.findall(rf"<{tag}>(.*?)</{tag}>", content, re.DOTALL))

SPLITS = {
    "train":             DATA_DIR / "train_sft_v2.jsonl",
    "eval":              DATA_DIR / "eval_sft_v2.jsonl",
    "train_interleaved": DATA_DIR / "train_interleaved.jsonl",
    "train_partB":       DATA_DIR / "train_partB.jsonl",
}

ALL_RECORDS = []   # full records — used by conversation renderer (Section 7)
rows = []

for split_name, path in SPLITS.items():
    if not path.exists():
        continue
    for rec in load_jsonl(path):
        meta = rec.get("metadata", {}).copy()
        msgs = rec.get("messages", [])
        asst = " ".join(m["content"] for m in msgs if m["role"] == "assistant")
        rows.append({
            **meta,
            "split":          split_name,
            "num_messages":   len(msgs),
            "response_chars": len(asst),
            "think_chars":    extract_tag_len(asst, "think"),
            "answer_chars":   extract_tag_len(asst, "answer"),
            "_idx":           len(ALL_RECORDS),
        })
        ALL_RECORDS.append(rec)

df = pd.DataFrame(rows)
df["category"]     = df["category"].fillna("(none)")
df["tool_profile"] = df["tool_profile"].fillna("(none)")
print(f"Loaded {len(df):,} records | {df['split'].nunique()} splits")
print(df.groupby("split").size().to_string())

## Section 2 — Dataset Overview

In [ ]:
# --- 2a: Summary metric cards ---
total      = len(df)
avg_score  = df["constitution_score"].dropna().mean() if "constitution_score" in df.columns else float("nan")
avg_len    = df["response_chars"].mean()
rev_pct    = df["revised"].dropna().mean() * 100 if "revised" in df.columns else float("nan")

metrics = [
    (f"{total:,}",            "Total Records"),
    (f"{df['split'].nunique()}", "Splits"),
    (f"{df['category'].nunique()}", "Categories"),
    (f"{avg_score:.3f}",      "Avg Score (train+eval)"),
    (f"{avg_len:,.0f}",       "Avg Response Length (chars)"),
    (f"{rev_pct:.0f}%",       "Revised (train+eval)"),
]
cards = "".join(f"""
  <div style="background:#f8fafc;border:1px solid #e2e8f0;border-radius:10px;
              padding:16px 24px;min-width:140px;text-align:center">
    <div style="font-size:26px;font-weight:700;color:#1e293b">{v}</div>
    <div style="font-size:12px;color:#64748b;margin-top:4px">{label}</div>
  </div>""" for v, label in metrics)
display(HTML(f"""
<div style="display:flex;gap:16px;flex-wrap:wrap;font-family:ui-sans-serif,sans-serif;
            margin:12px 0">{cards}</div>"""))

In [ ]:
# --- 2b: Category distribution donut ---
cat_counts = df["category"].value_counts().reset_index()
cat_counts.columns = ["category", "count"]
fig = px.pie(
    cat_counts, names="category", values="count",
    title="Training Data — Category Distribution",
    hole=0.45, color_discrete_sequence=PALETTE
)
fig.update_traces(textposition="outside", textinfo="label+percent")
fig.update_layout(showlegend=False, margin=dict(t=50, b=20))
save_fig(fig, "01_category_distribution")

In [ ]:
# --- 2c: Records per category per split (grouped bar) ---
split_cat = df.groupby(["split", "category"]).size().reset_index(name="count")
fig = px.bar(
    split_cat, x="category", y="count", color="split",
    barmode="group",
    title="Records per Category by Split",
    color_discrete_sequence=PALETTE,
    labels={"count": "Count", "category": "Category", "split": "Split"}
)
fig.update_xaxes(tickangle=-30)
save_fig(fig, "02_split_category_counts")

## Section 3 — Constitution Quality Analysis

In [ ]:
# --- 3a: Score distribution histogram ---
fig = px.histogram(
    df, x="constitution_score", nbins=20,
    title="Constitution Score Distribution",
    labels={"constitution_score": "Score (0–1)", "count": "Records"},
    color_discrete_sequence=[PALETTE[0]]
)
fig.update_layout(bargap=0.05)
save_fig(fig, "03_score_distribution")

In [ ]:
# --- 3b: Score by category ---
score_df = df[df["constitution_score"].notna()].copy()
fig = px.box(
    score_df, x="category", y="constitution_score",
    color="category", color_discrete_sequence=PALETTE,
    title="Constitution Score by Category",
    labels={"constitution_score": "Score", "category": "Category"},
    points="all"
)
fig.update_layout(showlegend=False)
fig.update_xaxes(tickangle=-30)
save_fig(fig, "04_score_by_category")

In [ ]:
# --- 3c: Draft violations histogram ---
viol_df = df[df["constitution_violations_in_draft"].notna()].copy()
fig = px.histogram(
    viol_df, x="constitution_violations_in_draft",
    title="Constitution Violations in Draft",
    labels={"constitution_violations_in_draft": "Violations", "count": "Records"},
    color_discrete_sequence=[PALETTE[1]]
)
fig.update_layout(bargap=0.1)
save_fig(fig, "05_violations_histogram")

In [ ]:
# --- 3d: Score vs violations scatter ---
sv_df = df[df["constitution_score"].notna() & df["constitution_violations_in_draft"].notna()].copy()
fig = px.scatter(
    sv_df, x="constitution_violations_in_draft", y="constitution_score",
    color="category",
    title="Constitution Score vs. Violations in Draft",
    labels={"constitution_violations_in_draft": "Violations in Draft",
            "constitution_score": "Final Score"},
    color_discrete_sequence=PALETTE,
    hover_data=["category", "tool_profile"]
)
save_fig(fig, "06_score_vs_violations")

## Section 4 — Tool Profile & Category Analysis

In [ ]:
# --- 4a: Tool profile distribution ---
tp_df = df[df["tool_profile"] != "(none)"].copy()
tp_counts = tp_df["tool_profile"].value_counts().reset_index()
tp_counts.columns = ["tool_profile", "count"]
fig = px.pie(
    tp_counts, names="tool_profile", values="count", hole=0.45,
    title="Tool Profile Distribution",
    color_discrete_sequence=PALETTE
)
fig.update_traces(textposition="outside", textinfo="label+percent")
fig.update_layout(showlegend=False)
save_fig(fig, "07_tool_profile_distribution")

In [ ]:
# --- 4b: Category × tool_profile count heatmap ---
heat_df = df[(df["category"] != "(none)") & (df["tool_profile"] != "(none)")].copy()
pivot = (
    heat_df.groupby(["category", "tool_profile"]).size()
    .reset_index(name="count")
    .pivot(index="category", columns="tool_profile", values="count")
    .fillna(0)
    .astype(int)
)
fig = go.Figure(go.Heatmap(
    z=pivot.values,
    x=pivot.columns.tolist(),
    y=pivot.index.tolist(),
    colorscale="Blues",
    text=pivot.values,
    texttemplate="%{text}",
    showscale=True,
))
fig.update_layout(
    title="Category × Tool Profile Count",
    xaxis_title="Tool Profile",
    yaxis_title="Category",
    margin=dict(l=160)
)
save_fig(fig, "08_category_toolprofile_heatmap")

In [ ]:
# --- 4c: Average constitution score by tool profile ---
scored_df = df[(df["tool_profile"] != "(none)") & df["constitution_score"].notna()].copy()
avg_tp = scored_df.groupby("tool_profile")["constitution_score"].mean().reset_index()
fig = px.bar(
    avg_tp, x="tool_profile", y="constitution_score",
    title="Average Constitution Score by Tool Profile",
    color="tool_profile", color_discrete_sequence=PALETTE,
    labels={"constitution_score": "Avg Score", "tool_profile": "Tool Profile"},
    text_auto=".3f"
)
fig.update_layout(showlegend=False, yaxis_range=[0.85, 1.0])
save_fig(fig, "09_avg_score_by_tool_profile")

## Section 5 — Response Quality Deep-Dive

In [ ]:
# --- 5a: Think-tag vs answer-tag length scatter ---
has_think = df[(df["think_chars"] > 0) & (df["category"] != "(none)")].copy()
fig = px.scatter(
    has_think,
    x="think_chars", y="answer_chars",
    color="category", size="constitution_score", size_max=14,
    title="Think-Tag Length vs Answer-Tag Length",
    labels={"think_chars": "<think> block (chars)", "answer_chars": "<answer> block (chars)"},
    color_discrete_sequence=PALETTE,
    hover_data=["tool_profile", "constitution_score", "constitution_violations_in_draft"]
)
save_fig(fig, "10_think_vs_answer_scatter")

In [ ]:
# --- 5b: Response length distribution by category ---
cat_df = df[df["category"] != "(none)"].copy()
fig = px.violin(
    cat_df, x="category", y="response_chars",
    color="category", box=True, points="outliers",
    title="Response Length Distribution by Category",
    labels={"response_chars": "Response Length (chars)", "category": "Category"},
    color_discrete_sequence=PALETTE
)
fig.update_layout(showlegend=False)
fig.update_xaxes(tickangle=-30)
save_fig(fig, "11_response_length_by_category")

In [ ]:
# --- 5c: Response length by tool profile ---
tp_r_df = df[df["tool_profile"] != "(none)"].copy()
fig = px.box(
    tp_r_df, x="tool_profile", y="response_chars",
    color="tool_profile", points="all",
    title="Response Length by Tool Profile",
    labels={"response_chars": "Response Length (chars)", "tool_profile": "Tool Profile"},
    color_discrete_sequence=PALETTE
)
fig.update_layout(showlegend=False)
save_fig(fig, "12_response_length_by_tool_profile")

In [ ]:
# --- 5d: Score vs response length ---
ols_df = df[df["constitution_score"].notna() & (df["category"] != "(none)")].copy()
fig = px.scatter(
    ols_df, x="response_chars", y="constitution_score",
    color="category",
    trendline="ols",
    title="Constitution Score vs. Response Length",
    labels={"response_chars": "Response Length (chars)", "constitution_score": "Score"},
    color_discrete_sequence=PALETTE,
    hover_data=["tool_profile"]
)
save_fig(fig, "13_score_vs_response_length")

## Section 6 — Cross-Model Performance Comparison

Compares base model vs custom (SFT-trained) model, with and without tools, on the same prompts.
Metrics extracted: number of turns, tool calls made, response length, reasoning depth (think-block length), and answer clarity (presence of `<answer>` tag).

In [ ]:
# --- 6a: Load reports and extract per-run metrics ---
def load_json_reports(pattern):
    return [json.load(open(p, encoding="utf-8"))
            for p in sorted(REPORTS_DIR.glob(pattern))]

benchmarks  = load_json_reports("benchmark_*.json")
comparisons = load_json_reports("comparison_*.json")
print(f"Benchmarks: {len(benchmarks)} | Comparisons: {len(comparisons)}")

def run_metrics(conversation):
    """Extract scalar metrics from a single model run conversation."""
    asst_msgs  = [m for m in conversation if m["role"] == "assistant"]
    tool_calls = sum(1 for m in asst_msgs if "<tool>" in m.get("content", ""))
    has_answer = sum(1 for m in asst_msgs if "<answer>" in m.get("content", ""))
    avg_think  = (sum(extract_tag_len(m["content"], "think") for m in asst_msgs)
                  / len(asst_msgs) if asst_msgs else 0)
    avg_len    = (sum(len(m["content"]) for m in asst_msgs)
                  / len(asst_msgs) if asst_msgs else 0)
    return {
        "turns":      len(asst_msgs),
        "tool_calls": tool_calls,
        "has_answer": has_answer,
        "avg_think":  avg_think,
        "avg_len":    avg_len,
    }

In [ ]:
# --- 6b: Turn count — base vs custom across all comparison prompts ---
comp_rows = []
for c in comparisons:
    prompt = c.get("prompt", "")
    short  = (prompt[:55] + "…") if len(prompt) > 55 else prompt
    bm     = run_metrics(c.get("base_model_no_tools", {}).get("conversation", []))
    cm     = run_metrics(c.get("custom_model_output", {}).get("conversation", []))
    comp_rows.append({
        "prompt":            short,
        "Base (no tools)":   bm["turns"],
        "Custom (w/ tools)": cm["turns"],
    })
cdf = pd.DataFrame(comp_rows)

fig = go.Figure([
    go.Bar(name="Base (no tools)",   x=cdf["prompt"], y=cdf["Base (no tools)"],   marker_color=PALETTE[0]),
    go.Bar(name="Custom (w/ tools)", x=cdf["prompt"], y=cdf["Custom (w/ tools)"], marker_color=PALETTE[1]),
])
fig.update_layout(
    barmode="group",
    title="Response Turn Count: Base vs Custom Model",
    xaxis_title="Prompt", yaxis_title="Turns",
    xaxis_tickangle=-40, legend_title="Model", height=420
)
save_fig(fig, "14_turn_count_comparison")

In [ ]:
# --- 6c: Multi-metric radar — aggregate comparison across all comparisons ---
def avg_metrics(key):
    vals = []
    for c in comparisons:
        conv = c.get(key, {}).get("conversation", [])
        if conv:
            vals.append(run_metrics(conv))
    if not vals:
        return {k: 0 for k in ["turns", "tool_calls", "has_answer", "avg_think", "avg_len"]}
    return pd.DataFrame(vals).mean().to_dict()

base_avg   = avg_metrics("base_model_no_tools")
custom_avg = avg_metrics("custom_model_output")

metrics_keys   = ["turns", "tool_calls", "has_answer", "avg_think", "avg_len"]
metrics_labels = ["Turns", "Tool Calls", "Answer Tags", "Avg Think (chars)", "Avg Resp (chars)"]

def normalise(vals):
    max_v = max(abs(v) for v in vals) or 1
    return [v / max_v for v in vals]

base_n   = normalise([base_avg[k]   for k in metrics_keys])
custom_n = normalise([custom_avg[k] for k in metrics_keys])

fig = go.Figure()
for label, vals, color in [
    ("Base (no tools)",   base_n,   PALETTE[0]),
    ("Custom (w/ tools)", custom_n, PALETTE[1]),
]:
    fig.add_trace(go.Scatterpolar(
        r=vals + [vals[0]],
        theta=metrics_labels + [metrics_labels[0]],
        fill="toself", name=label, line_color=color,
    ))
fig.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
    title="Model Capability Radar — Normalised Metrics",
    showlegend=True,
)
save_fig(fig, "15_model_radar")

In [ ]:
# --- 6d: Per-prompt metric table ---
table_rows = []
for c in comparisons:
    prompt = c.get("prompt", "")
    short  = (prompt[:60] + "…") if len(prompt) > 60 else prompt
    bm     = run_metrics(c.get("base_model_no_tools", {}).get("conversation", []))
    cm     = run_metrics(c.get("custom_model_output", {}).get("conversation", []))
    table_rows.append({
        "Prompt":             short,
        "Base Turns":         bm["turns"],
        "Custom Turns":       cm["turns"],
        "Custom Tool Calls":  cm["tool_calls"],
        "Custom Answer Tags": cm["has_answer"],
        "Base Avg Len":       f"{bm['avg_len']:.0f}",
        "Custom Avg Len":     f"{cm['avg_len']:.0f}",
    })
pd.DataFrame(table_rows).style.background_gradient(
    subset=["Base Turns", "Custom Turns", "Custom Tool Calls"], cmap="Blues"
)

In [ ]:
# --- 6e: Benchmark multi-run comparison ---
for bm in benchmarks:
    run_rows = []
    for run_name, run_data in bm.get("runs", {}).items():
        m = run_metrics(run_data.get("conversation", []))
        run_rows.append({"Run": run_name, **m})
    if not run_rows:
        continue
    bdf = pd.DataFrame(run_rows)
    colors = PALETTE[:len(bdf)]
    fig = make_subplots(rows=1, cols=3,
                        subplot_titles=("Turns", "Tool Calls", "Avg Response Length (chars)"))
    fig.add_trace(go.Bar(x=bdf["Run"], y=bdf["turns"],      marker_color=colors, showlegend=False), row=1, col=1)
    fig.add_trace(go.Bar(x=bdf["Run"], y=bdf["tool_calls"],  marker_color=colors, showlegend=False), row=1, col=2)
    fig.add_trace(go.Bar(x=bdf["Run"], y=bdf["avg_len"],     marker_color=colors, showlegend=False), row=1, col=3)
    ts = bm.get("timestamp", "")
    fig.update_layout(title=f"Benchmark Multi-Run Comparison — {ts}", height=380)
    save_fig(fig, f"16_benchmark_{ts}")

## Section 7 — Conversation Viewer

Light-mode, dissertation-ready conversation renderer with syntax highlighting for `<think>`, `<answer>`, and `<tool>` tags. Use the interactive browser to explore training samples by category and tool profile.

In [ ]:
# --- 7a: Conversation renderer ---
_ROLE_STYLE = {
    "system":    ("System",    "#eff6ff", "#1d4ed8"),
    "user":      ("User",      "#f0fdf4", "#16a34a"),
    "assistant": ("Assistant", "#faf5ff", "#7c3aed"),
}

def _highlight_tags(text):
    text = re.sub(
        r"<think>(.*?)</think>",
        lambda m: (
            "<details open style='margin:4px 0'>"
            "<summary style='color:#6366f1;font-weight:600;cursor:pointer'>&#x1f4ad; Reasoning</summary>"
            "<pre style='white-space:pre-wrap;background:#f8f7ff;padding:10px;border-radius:6px;"
            "font-size:13px;color:#3730a3;margin:4px 0'>"
            + m.group(1) + "</pre></details>"
        ),
        text, flags=re.DOTALL
    )
    text = re.sub(
        r"<answer>(.*?)</answer>",
        lambda m: (
            "<div style='background:#f0fdf4;border-left:3px solid #22c55e;"
            "padding:8px 12px;margin:6px 0;border-radius:0 6px 6px 0'>"
            "<span style='font-weight:600;color:#16a34a'>Answer: </span>"
            + m.group(1) + "</div>"
        ),
        text, flags=re.DOTALL
    )
    text = re.sub(
        r"<tool>(.*?)</tool>",
        lambda m: (
            "<code style='background:#fff7ed;border:1px solid #fed7aa;"
            "padding:3px 8px;border-radius:4px;font-size:13px'>"
            "&#x1f527; " + m.group(1) + "</code>"
        ),
        text, flags=re.DOTALL
    )
    return text

def render_conversation(messages, title="Conversation", score=None, category=None):
    meta = ""
    if category or score is not None:
        parts = []
        if category:          parts.append(f"<strong>Category:</strong> {category}")
        if score is not None: parts.append(f"<strong>Score:</strong> {score:.3f}")
        meta = f"<p style='font-size:12px;color:#64748b;margin:4px 0 10px'>{' · '.join(parts)}</p>"

    bubbles = ""
    for msg in messages:
        role           = msg.get("role", "assistant")
        label, bg, acc = _ROLE_STYLE.get(role, ("?", "#fff", "#000"))
        content        = _highlight_tags(msg.get("content", ""))
        bubbles += (
            f"<div style='background:{bg};border:1px solid #e2e8f0;border-radius:8px;"
            f"padding:10px 14px'>"
            f"<div style='font-size:11px;font-weight:700;color:{acc};"
            f"text-transform:uppercase;letter-spacing:.06em;margin-bottom:6px'>{label}</div>"
            f"<div style='font-size:14px;color:#1e293b;line-height:1.6'>{content}</div>"
            f"</div>"
        )

    html = (
        f"<div style='font-family:ui-sans-serif,system-ui,sans-serif;max-width:860px;"
        f"border:1px solid #e2e8f0;border-radius:12px;overflow:hidden;"
        f"box-shadow:0 1px 4px rgba(0,0,0,0.06)'>"
        f"<div style='background:#f8fafc;padding:12px 18px;border-bottom:1px solid #e2e8f0'>"
        f"<h3 style='margin:0;font-size:16px;color:#1e293b'>{title}</h3>{meta}</div>"
        f"<div style='padding:14px 16px;display:flex;flex-direction:column;gap:10px'>"
        f"{bubbles}</div></div>"
    )
    return HTML(html)

In [ ]:
# --- 7b: Interactive training sample browser ---
from ipywidgets import interact, Dropdown, IntSlider

_CATEGORIES    = sorted(df[df["category"] != "(none)"]["category"].unique().tolist())
_TOOL_PROFILES = ["(any)"] + sorted(df[df["tool_profile"] != "(none)"]["tool_profile"].unique().tolist())

def browse_samples(category=_CATEGORIES[0], tool_profile="(any)", seed=42):
    subset = df[df["category"] == category]
    if tool_profile != "(any)":
        subset = subset[subset["tool_profile"] == tool_profile]
    if subset.empty:
        display(HTML("<p style='color:#ef4444;font-family:sans-serif'>No matching records.</p>"))
        return
    row = subset.sample(1, random_state=seed).iloc[0]
    rec = ALL_RECORDS[row["_idx"]]
    display(render_conversation(
        rec["messages"],
        title=f"{row['category']} · {row['tool_profile']}",
        score=row.get("constitution_score"),
        category=row["category"],
    ))

interact(
    browse_samples,
    category=Dropdown(options=_CATEGORIES, description="Category:"),
    tool_profile=Dropdown(options=_TOOL_PROFILES, description="Tool Profile:"),
    seed=IntSlider(min=0, max=100, value=42, description="Seed:"),
)

In [ ]:
# --- 7c: Side-by-side comparison viewer ---
from ipywidgets import interact, Dropdown

_COMP_LABELS = [
    ((c.get("prompt", "")[:60] + "…") if len(c.get("prompt", "")) > 60 else c.get("prompt", ""))
    for c in comparisons
]

def show_comparison(prompt_label=_COMP_LABELS[0] if _COMP_LABELS else ""):
    idx = _COMP_LABELS.index(prompt_label) if prompt_label in _COMP_LABELS else 0
    c   = comparisons[idx]
    display(HTML(
        f"<h3 style='font-family:sans-serif;margin:12px 0 4px'>"
        f"Prompt: <em style='font-weight:400'>{c.get('prompt', '')}</em></h3>"
    ))
    display(render_conversation(
        c["base_model_no_tools"]["conversation"], "Base Model (No Tools)"))
    display(HTML("<br>"))
    display(render_conversation(
        c["custom_model_output"]["conversation"], "Custom Model (With Tools)"))

if _COMP_LABELS:
    interact(show_comparison, prompt_label=Dropdown(options=_COMP_LABELS, description="Prompt:"))
else:
    display(HTML("<p style='color:#64748b'>No comparison reports found in reports/</p>"))

## Section 8 — ROUGE Scores

Compares ROUGE-1, ROUGE-2, ROUGE-L F1 across checkpoints for two reference sources: eval split gold responses and constitution probe baseline. Load `reports/rouge_*.json` produced by `2_model_trainer.py publish()`.

In [ ]:
# --- 8a: Load ROUGE reports ---
rouge_reports = []
for p in sorted(REPORTS_DIR.glob("rouge_*.json")):
    with open(p, encoding="utf-8") as f:
        rouge_reports.append(json.load(f))

print(f"ROUGE reports: {len(rouge_reports)}")
for r in rouge_reports:
    ev  = "✓" if r.get("eval_split_rouge")     else "✗"
    pb  = "✓" if r.get("probe_baseline_rouge") else "✗"
    rwd = f"{r['grpo_held_out_reward']:.4f}" if r.get("grpo_held_out_reward") is not None else "—"
    print(f"  {r['checkpoint']:35}  eval={ev}  probe={pb}  grpo_reward={rwd}")

In [ ]:
# --- 8b: ROUGE F1 — eval split gold responses ---
def _rouge_to_df(reports, key):
    rows = []
    for r in reports:
        block = r.get(key)
        if not block:
            continue
        rows.append({
            "checkpoint": r["checkpoint"],
            "ROUGE-1":    block["rouge1"]["fmeasure"],
            "ROUGE-2":    block["rouge2"]["fmeasure"],
            "ROUGE-L":    block["rougeL"]["fmeasure"],
        })
    return pd.DataFrame(rows)

eval_df = _rouge_to_df(rouge_reports, "eval_split_rouge")

if not eval_df.empty:
    melted = eval_df.melt(id_vars="checkpoint", var_name="Metric", value_name="F1")
    fig = px.bar(
        melted, x="checkpoint", y="F1", color="Metric", barmode="group",
        title="ROUGE F1 — Eval Split (Gold Responses)",
        color_discrete_sequence=PALETTE, text_auto=".3f",
        labels={"checkpoint": "Checkpoint", "F1": "F1 Score"},
    )
    fig.update_layout(yaxis_range=[0, 1])
    save_fig(fig, "17_rouge_eval_split")
else:
    print("No eval-split ROUGE data yet — run 2_model_trainer.py to generate reports/rouge_*.json")

In [ ]:
# --- 8c: ROUGE F1 — probe baseline (constitution drift indicator) ---
probe_df = _rouge_to_df(rouge_reports, "probe_baseline_rouge")

if not probe_df.empty:
    melted = probe_df.melt(id_vars="checkpoint", var_name="Metric", value_name="F1")
    fig = px.bar(
        melted, x="checkpoint", y="F1", color="Metric", barmode="group",
        title="ROUGE F1 — Probe Baseline (Constitution Drift Indicator)",
        color_discrete_sequence=PALETTE, text_auto=".3f",
        labels={"checkpoint": "Checkpoint", "F1": "F1 Score"},
    )
    fig.update_layout(yaxis_range=[0, 1])
    save_fig(fig, "18_rouge_probe_baseline")
else:
    print("No probe-baseline ROUGE data yet.")

In [ ]:
# --- 8d: GRPO held-out reward ---
reward_rows = [
    {"checkpoint": r["checkpoint"], "Held-out Reward": r["grpo_held_out_reward"]}
    for r in rouge_reports
    if r.get("grpo_held_out_reward") is not None
]
if reward_rows:
    rdf = pd.DataFrame(reward_rows)
    fig = px.bar(
        rdf, x="checkpoint", y="Held-out Reward",
        title="GRPO Held-Out Reward Score (10% held-out prompts, full reward function)",
        color_discrete_sequence=[PALETTE[2]], text_auto=".4f",
        labels={"checkpoint": "Checkpoint"},
    )
    fig.update_layout(yaxis_range=[0, 1])
    save_fig(fig, "19_grpo_held_out_reward")
else:
    print("No GRPO held-out reward data yet — run GRPO training to generate.")

## Section 9 — Training Loss Curves

SFT loss curves loaded from `models/checkpoint_sft*/loss_history.json`. GRPO reward/loss curves loaded from `models/checkpoint_grpo*/grpo_loss_history.json`.

In [ ]:
# --- 9a: Discover and load loss/reward history files ---
models_dir = Path("models")

sft_histories  = {}
grpo_histories = {}

for p in sorted(models_dir.glob("checkpoint_sft*/loss_history.json")):
    with open(p, encoding="utf-8") as f:
        sft_histories[p.parent.name] = json.load(f)

for p in sorted(models_dir.glob("checkpoint_grpo*/grpo_loss_history.json")):
    with open(p, encoding="utf-8") as f:
        grpo_histories[p.parent.name] = json.load(f)

print(f"SFT checkpoints:  {list(sft_histories.keys())  or ['none found']}")
print(f"GRPO checkpoints: {list(grpo_histories.keys()) or ['none found']}")

In [ ]:
# --- 9b: SFT train + eval loss curves ---
for label, history in sft_histories.items():
    train_rows = [h for h in history if "loss" in h and "eval_loss" not in h]
    eval_rows  = [h for h in history if "eval_loss" in h]

    if not train_rows:
        print(f"  {label}: no training-loss entries found in history")
        continue

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=[h["step"] for h in train_rows],
        y=[h["loss"]  for h in train_rows],
        mode="lines", name="Train Loss", line_color=PALETTE[0],
    ))
    if eval_rows:
        fig.add_trace(go.Scatter(
            x=[h["step"]      for h in eval_rows],
            y=[h["eval_loss"] for h in eval_rows],
            mode="lines+markers", name="Eval Loss", line_color=PALETTE[1],
        ))
    fig.update_layout(
        title=f"SFT Loss Curves — {label}",
        xaxis_title="Step", yaxis_title="Cross-Entropy Loss",
        legend_title="Split",
    )
    safe = label.replace("/", "_")
    save_fig(fig, f"20_sft_loss_{safe}")

In [ ]:
# --- 9c: GRPO policy loss + mean reward curves ---
for label, history in grpo_histories.items():
    loss_rows   = [h for h in history if "loss" in h]
    reward_rows = [h for h in history if "rewards/mean" in h or "reward" in h]

    if not loss_rows and not reward_rows:
        print(f"  {label}: no usable entries in grpo_loss_history.json")
        continue

    fig = make_subplots(rows=1, cols=2, subplot_titles=("Policy Loss", "Mean Reward"))

    if loss_rows:
        fig.add_trace(go.Scatter(
            x=[h["step"] for h in loss_rows],
            y=[h["loss"] for h in loss_rows],
            mode="lines", name="Policy Loss", line_color=PALETTE[0], showlegend=True,
        ), row=1, col=1)

    if reward_rows:
        rkey = "rewards/mean" if "rewards/mean" in reward_rows[0] else "reward"
        fig.add_trace(go.Scatter(
            x=[h["step"]  for h in reward_rows],
            y=[h[rkey]    for h in reward_rows],
            mode="lines", name="Mean Reward", line_color=PALETTE[2], showlegend=True,
        ), row=1, col=2)

    fig.update_layout(title=f"GRPO Training Curves — {label}", height=420)
    safe = label.replace("/", "_")
    save_fig(fig, f"21_grpo_curves_{safe}")